This notebook fits orthos to all replicates of the shendure-calibrated simulations

Imports

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

2025-10-10 11:18:26.148752: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-10 11:18:26.153153: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

Autoreload for dev

In [2]:
%load_ext autoreload
%autoreload 2

Create a nice large cluster. We will need the resorces.

In [3]:
cluster=SLURMCluster(
    cores=4,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=2,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=4:00:00",
        f"--output=slave_%j.out"]
)
cluster.scale(jobs=4)
client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="20s"  # Worker heartbeat interval
    )

Load the simulation object

In [4]:
DATA_ROOT="/gpfs/gibbs/pi/reilly/tabula_data"
simu_obj_cell=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251008")

In [5]:
#temporary : obj created w/ older version of code, necessitating this. 
simu_obj_cell.orthos=[]

Fit orthos to all replicates

In [6]:
simu_obj_cell.create_orthos_for_all_replicates(client)

Save

In [12]:
x=simu_obj_cell.orthos[0].result()

In [19]:
x.by_cre.model["Col1a2_chr6_77"]

AttributeError: 'NoneType' object has no attribute 'type'

AttributeError: 'NoneType' object has no attribute 'type'

In [7]:
simu_obj_cell.save(path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_with_orthos_20251008")

scMPRAforge: WARNING: Directory /gpfs/gibbs/pi/reilly/tabula_data/simulated/shendure_calibrated_sim_with_orthos_20251008 already exists, continuing.
scMPRAforge: WARNING: Directory /gpfs/gibbs/pi/reilly/tabula_data/simulated/shendure_calibrated_sim_with_orthos_20251008/orthos already exists, continuing.


RuntimeError: <class 'distributed.client.Future'> object not properly initialized. This can happen if the object is being deserialized outside of the context of a Client or Worker.

In [ ]:
simu_obj_cell.orthos[0].result()

Shut down the cluster

In [ ]:
client.close()
cluster.close()

# Move stuff below to its own nb...

Load the object twice, one for each set of hypothesies.

In [ ]:
DATA_ROOT="/gpfs/gibbs/pi/reilly/tabula_data"
simu_obj_cell_type_hypotheses=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251008")
simu_obj_cre_hypotheses=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251008")

Load the original ortho

In [ ]:
primordial=scm.ortho.load(client,f"{DATA_ROOT}/shendure","ortho_primordial_v3")

Let's start with the by cell type hypotheses

Create the hypothesis set

In [ ]:
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=primordial.training_data,
    reference_cre="reference",
    meta="emvar_screen",
)

Run wald on all reps

In [ ]:
simu_obj_cell_type_hypotheses._test_all_replicates(client,hs_all_ct,test="wald")